#### Bước 1: Cài thư viện

In [1]:
%pip install selenium beautifulsoup4

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Bước 2: Mở trang danh sách

In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup
import csv
import json
import os
import re
import time

BASE_DOMAIN = "https://batdongsan.com.vn"
SOURCES = {
    "can_ho_chung_cu": {
        "url": f"{BASE_DOMAIN}/ban-can-ho-chung-cu-tp-hcm",
        "target": 2000,
    },
    "chung_cu_mini_can_ho_dich_vu": {
        "url": f"{BASE_DOMAIN}/ban-can-ho-chung-cu-mini-tp-hcm",
        "target": 1000,
    },
    "nha_rieng": {
        "url": f"{BASE_DOMAIN}/ban-nha-rieng-tp-hcm",
        "target": 1000,
    },
    "nha_biet_thu_lien_ke": {
        "url": f"{BASE_DOMAIN}/ban-nha-biet-thu-lien-ke-tp-hcm",
        "target": 1000,
    },
}
CHECKPOINT_DIR = "batdongsan_checkpoints"
CHECKPOINT_EVERY = 20
OUTPUT_FILE = "batdongsan_hcm.csv"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)


def create_driver():
    options = webdriver.ChromeOptions()
    options.add_argument("--start-maximized")
    options.page_load_strategy = "eager"

    driver = webdriver.Chrome(options=options)
    driver.set_page_load_timeout(20)
    return driver


def page_url(base_url, page_number):
    if page_number <= 1:
        return base_url
    return f"{base_url}/p{page_number}"


def checkpoint_path(category):
    return os.path.join(CHECKPOINT_DIR, f"{category}.json")


def load_checkpoint(category):
    path = checkpoint_path(category)
    if not os.path.exists(path):
        return {"results": [], "page": 1, "card_index": 0}

    with open(path, "r", encoding="utf-8") as checkpoint_file:
        checkpoint = json.load(checkpoint_file)

    return {
        "results": checkpoint.get("results", []),
        "page": max(1, int(checkpoint.get("page", 1))),
        "card_index": max(0, int(checkpoint.get("card_index", 0))),
    }


def save_checkpoint(category, results, page, card_index):
    path = checkpoint_path(category)
    temporary_file = f"{path}.tmp"
    checkpoint = {
        "results": results,
        "page": page,
        "card_index": card_index,
    }

    with open(temporary_file, "w", encoding="utf-8") as checkpoint_file:
        json.dump(checkpoint, checkpoint_file, ensure_ascii=False, indent=2)

    os.replace(temporary_file, path)


print("Nguon:", len(SOURCES))
print("Muc tieu tong:", sum(item["target"] for item in SOURCES.values()), "tin")
print("Checkpoint moi:", CHECKPOINT_EVERY, "tin")


Nguon: 4
Muc tieu tong: 5000 tin
Checkpoint moi: 20 tin


## Chế độ test 50 tin

Cell kế tiếp chỉ dùng để chạy test theo tỷ lệ 20/10/10/10. Đặt `RUN_TEST = True` khi cần test; mặc định `False` để crawl đủ 5.000 tin.

In [26]:
RUN_TEST = False

TEST_SOURCES = {
    category: {
        "url": source["url"],
        "target": target,
    }
    for category, source, target in [
        ("can_ho_chung_cu", SOURCES["can_ho_chung_cu"], 20),
        ("chung_cu_mini_can_ho_dich_vu", SOURCES["chung_cu_mini_can_ho_dich_vu"], 10),
        ("nha_rieng", SOURCES["nha_rieng"], 10),
        ("nha_biet_thu_lien_ke", SOURCES["nha_biet_thu_lien_ke"], 10),
    ]
}

if RUN_TEST:
    SOURCES = TEST_SOURCES
    CHECKPOINT_DIR = "batdongsan_checkpoints_test"
    OUTPUT_FILE = "data_test.csv"
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    print("CHE DO TEST:", sum(item["target"] for item in SOURCES.values()), "tin")
    print("Ty le muc tieu:", {category: item["target"] for category, item in SOURCES.items()})
    print("Checkpoint:", CHECKPOINT_DIR)
    print("Output:", OUTPUT_FILE)
else:
    CHECKPOINT_DIR = "batdongsan_checkpoints"
    OUTPUT_FILE = "batdongsan_data_raw.csv"
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    print("Bo qua che do test; giu cau hinh crawl 5000 tin.")
    print("Output:", OUTPUT_FILE)


Bo qua che do test; giu cau hinh crawl 5000 tin.
Output: batdongsan_data_raw.csv


In [25]:
if RUN_TEST:
    for category in SOURCES:
        test_checkpoint = checkpoint_path(category)
        if os.path.exists(test_checkpoint):
            os.remove(test_checkpoint)

    print("Da xoa checkpoint test cu; se crawl lai tu trang 1/card 0.")
else:
    print("Bo qua xoa checkpoint test.")


Bo qua xoa checkpoint test.


### Bước 3: Tìm các tin đăng

In [12]:
print("Bo qua buoc tim card rieng le; vong crawl se tai card theo tung trang.")

Bo qua buoc tim card rieng le; vong crawl se tai card theo tung trang.


### Bước 4: Parser theo các trường 

In [4]:
def clean_text(text):
    if text is None:
        return None
    text = re.sub(r"[\u200e\u200f\u202a-\u202e\u2066-\u2069]", "", text)
    return re.sub(r"\s+", " ", text.replace("\xa0", " ")).strip()


def find_first(pattern, text):
    match = re.search(pattern, text or "", re.IGNORECASE)
    return clean_text(match.group(1)) if match else None


def absolute_url(url):
    if url and url.startswith("/"):
        return BASE_DOMAIN + url
    return url


def parse_card(card, category):
    text = clean_text(card.get_text(" ", strip=True)) or ""
    ma_tin = card.get("data-product-id")

    tieu_de = None
    if card.name == "a" and card.get("title"):
        tieu_de = card.get("title")
    if not tieu_de:
        title_link = card.find("a", attrs={"title": True})
        if title_link:
            tieu_de = title_link.get("title")
    if not tieu_de:
        title_heading = card.find(
            ["h2", "h3"],
            class_=re.compile(r"(?:title|name)", re.IGNORECASE),
        )
        if title_heading:
            tieu_de = title_heading.get_text(" ", strip=True)
    tieu_de = clean_text(tieu_de)

    url = None
    if card.name == "a" and card.get("href"):
        url = card.get("href")
    if not url:
        link_tag = card.find("a", href=re.compile(r"-pr\d+", re.IGNORECASE))
        if link_tag:
            url = link_tag.get("href")
    url = absolute_url(url)

    published_tag = card.select_one("span.re__card-published-info-published-at")
    ngay_dang = None
    if published_tag:
        ngay_dang = published_tag.get("aria-label") or published_tag.get_text(" ", strip=True)
        ngay_dang = clean_text(ngay_dang)

    agent_tag = card.select_one(
        "div.agent-name.agent-item, .entry-agent-infor .agent-name"
    )
    ten_moi_gioi_san = clean_text(agent_tag.get_text(" ", strip=True)) if agent_tag else None
    if not ten_moi_gioi_san:
        contact_tag = card.select_one("div.re__card-contact")
        if contact_tag:
            agent_button = contact_tag.select_one("[data-kyc-name]")
            if agent_button:
                ten_moi_gioi_san = clean_text(agent_button.get("data-kyc-name"))

    gia = find_first(r"\b(\d+(?:[.,]\d+)?\s*(?:tỷ|ty|triệu|trieu))\b", text)
    dien_tich = find_first(r"\b(\d+(?:[.,]\d+)?\s*m²)\b", text)
    so_phong_ngu = find_first(r"\b(\d+)\s*PN\b", text)
    so_tang = find_first(r"\b(\d+)\s*tầng\b", text)
    huong_nha = find_first(r"(Đông Nam|Đông Bắc|Tây Nam|Tây Bắc|Đông|Tây|Nam|Bắc)", text)
    quan_huyen = find_first(r"\b((?:Quận|Huyện)\s+\d+(?:\s*\([^)]*\))?)", text)
    if not quan_huyen:
        quan_huyen = find_first(r"\b((?:Quận|Huyện)\s+[A-Za-zÀ-ỹ0-9 .-]+?)(?=\s*\(|\s*[·|]|$)", text)
    phuong_xa = find_first(r"\b((?:P\.|Phường|Xã|Thị trấn)\s*[A-Za-zÀ-ỹ0-9 .-]+)", text)

    legal_status = None
    for pattern in ["sổ hồng", "sổ đỏ", "pháp lý đầy đủ", "pháp lý rõ ràng", "đã có sổ", "giấy tờ đầy đủ"]:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            legal_status = clean_text(match.group(0))
            break

    description_candidates = [
        clean_text(div.get_text(" ", strip=True))
        for div in card.find_all("div")
    ]
    description_candidates = [item for item in description_candidates if item and len(item) > 80]

    return {
        "ma_tin": ma_tin,
        "tieu_de": tieu_de,
        "loai_hinh": category,
        "gia": gia,
        "dien_tich": dien_tich,
        "so_phong_ngu": so_phong_ngu,
        "so_tang": so_tang,
        "huong_nha": huong_nha,
        "tinh_thanh": "Hồ Chí Minh" if re.search(r"Hồ Chí Minh|TP\.?\s*HCM|TP\.?\s*Hồ Chí Minh", text, re.IGNORECASE) else None,
        "quan_huyen": quan_huyen,
        "phuong_xa": phuong_xa,
        "tinh_trang_phap_ly": legal_status,
        "ngay_dang": ngay_dang,
        "ten_moi_gioi_san": ten_moi_gioi_san,
        "nguon": "batdongsan.com.vn",
        "khu_vuc": quan_huyen,
        "dia_diem": None,
        "mo_ta": max(description_candidates, key=len) if description_candidates else None,
        "nguoi_dang": None,
        "thoi_gian_dang": ngay_dang,
        "so_phong": None,
        "url": url,
    }


### Bước 5: Cào toàn bộ card

In [5]:
cloudflare_keywords = [
    "just a moment",
    "performing security verification",
    "verify you are human",
    "security service to protect",
]
MAX_NO_NEW_PAGES = 3

all_results = []
carry_over = 0

for category, source in SOURCES.items():
    checkpoint = load_checkpoint(category)
    results = checkpoint["results"]
    page = checkpoint["page"]
    card_index = checkpoint["card_index"]
    target = source["target"] + carry_over
    requested_target = target
    seen_keys = {
        str(item.get("ma_tin") or item.get("url") or item.get("tieu_de"))
        for item in results
    }
    no_new_pages = 0
    category_finished = False

    print(f"\n===== {category}: {len(results)}/{target} =====")
    if carry_over:
        print(f"[{category}] Nhan bu {carry_over} tin tu loai hinh truoc.")

    while len(results) < target and not category_finished:
        driver = None
        batch_count = 0
        recovery_required = False

        try:
            driver = create_driver()

            while batch_count < CHECKPOINT_EVERY and len(results) < target:
                current_url = page_url(source["url"], page)
                print(f"[{category}] Mo trang {page}: {current_url}")

                try:
                    driver.get(current_url)
                    time.sleep(5)
                    title = driver.title.lower()
                    body_text = driver.find_element(By.TAG_NAME, "body").text.lower()
                except Exception as error:
                    print(f"[{category}] Loi tai trang: {error}")
                    recovery_required = True
                    break

                if any(keyword in title or keyword in body_text for keyword in cloudflare_keywords):
                    print(f"[{category}] Cloudflare dang chan.")
                    recovery_required = True
                    break

                soup = BeautifulSoup(driver.page_source, "html.parser")
                cards = soup.find_all(attrs={"data-product-id": True})

                if not cards:
                    no_new_pages += 1
                    print(f"[{category}] Trang {page} khong co card ({no_new_pages}/{MAX_NO_NEW_PAGES}).")
                    if no_new_pages >= MAX_NO_NEW_PAGES:
                        category_finished = True
                        print(f"[{category}] Da het tin hoac trang khong con du lieu.")
                        break
                    page += 1
                    card_index = 0
                    continue

                if card_index >= len(cards):
                    page += 1
                    card_index = 0
                    continue

                page_new_count = 0
                while card_index < len(cards) and batch_count < CHECKPOINT_EVERY and len(results) < target:
                    card = cards[card_index]
                    card_index += 1

                    if card.select_one("div.re__expired-overlay") is not None:
                        print(f"[{category}] Bo qua tin het han.")
                        continue

                    try:
                        data = parse_card(card, category)
                        key = str(data.get("ma_tin") or data.get("url") or data.get("tieu_de"))
                        if key == "None" or key in seen_keys:
                            continue

                        results.append(data)
                        seen_keys.add(key)
                        batch_count += 1
                        page_new_count += 1
                        print(f"[{category}] [{len(results)}/{target}] OK - {data['ma_tin']}")

                        if len(results) % CHECKPOINT_EVERY == 0:
                            save_checkpoint(category, results, page, card_index)
                            print(f"[{category}] Da checkpoint {len(results)} tin.")
                    except Exception as error:
                        print(f"[{category}] LOI parse card: {error}")

                if page_new_count == 0:
                    no_new_pages += 1
                    print(f"[{category}] Trang {page} khong co tin hop le moi ({no_new_pages}/{MAX_NO_NEW_PAGES}).")
                    if no_new_pages >= MAX_NO_NEW_PAGES:
                        category_finished = True
                        print(f"[{category}] Da het tin hop le moi; dung category.")
                        break
                else:
                    no_new_pages = 0

                if card_index >= len(cards):
                    page += 1
                    card_index = 0

        finally:
            save_checkpoint(category, results, page, card_index)
            print(f"[{category}] Checkpoint: {len(results)} tin | trang {page} | card {card_index}")

            if driver is not None:
                driver.quit()
                print(f"[{category}] Da dong browser.")

            if recovery_required:
                print(f"[{category}] Nghi 60 giay roi mo browser lai tu trang/card hien tai.")
                time.sleep(60)
            elif not category_finished and len(results) < target:
                time.sleep(2)

    all_results.extend(results[:target])
    carry_over = max(0, target - len(results))
    print(f"[{category}] Hoan tat: {len(results[:target])}/{requested_target}")
    if carry_over:
        print(f"[{category}] Con thieu {carry_over} tin; se chuyen sang loai hinh ke tiep.")

if carry_over:
    print(f"Con thieu tong cong {carry_over} tin sau khi da het cac loai hinh.")

df_result = all_results
print("\n==========================")
print("SO TIN DA CAO:", len(df_result))
print("==========================")
print(df_result[:10])



===== can_ho_chung_cu: 2000/2000 =====
[can_ho_chung_cu] Hoan tat: 2000/2000

===== chung_cu_mini_can_ho_dich_vu: 32/1000 =====
[chung_cu_mini_can_ho_dich_vu] Mo trang 1378: https://batdongsan.com.vn/ban-can-ho-chung-cu-mini-tp-hcm/p1378
[chung_cu_mini_can_ho_dich_vu] Bo qua tin het han.
[chung_cu_mini_can_ho_dich_vu] Bo qua tin het han.
[chung_cu_mini_can_ho_dich_vu] Bo qua tin het han.
[chung_cu_mini_can_ho_dich_vu] Bo qua tin het han.
[chung_cu_mini_can_ho_dich_vu] Bo qua tin het han.
[chung_cu_mini_can_ho_dich_vu] Bo qua tin het han.
[chung_cu_mini_can_ho_dich_vu] Bo qua tin het han.
[chung_cu_mini_can_ho_dich_vu] Bo qua tin het han.
[chung_cu_mini_can_ho_dich_vu] Bo qua tin het han.
[chung_cu_mini_can_ho_dich_vu] Bo qua tin het han.
[chung_cu_mini_can_ho_dich_vu] Trang 1378 khong co tin hop le moi (1/3).
[chung_cu_mini_can_ho_dich_vu] Mo trang 1379: https://batdongsan.com.vn/ban-can-ho-chung-cu-mini-tp-hcm/p1379
[chung_cu_mini_can_ho_dich_vu] Cloudflare dang chan.
[chung_cu_mini_

There was an error managing chromedriver (error sending request for url (https://googlechromelabs.github.io/chrome-for-testing/known-good-versions-with-downloads.json)); using driver found in the cache


[nha_rieng] Mo trang 38: https://batdongsan.com.vn/ban-nha-rieng-tp-hcm/p38
[nha_rieng] Loi tai trang: Message: unknown error: net::ERR_INTERNET_DISCONNECTED
  (Session info: chrome=153.0.8010.52)
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff77a8f1035+5a95]
	chromedriver!(No symbol) [0x7ff77a812c70]
	chromedriver!(No symbol) [0x7ff77a634a7d]
	chromedriver!(No symbol) [0x7ff77a63186b]
	chromedriver!(No symbol) [0x7ff77a621da1]
	chromedriver!(No symbol) [0x7ff77a623be4]
	chromedriver!(No symbol) [0x7ff77a622334]
	chromedriver!(No symbol) [0x7ff77a621b0b]
	chromedriver!(No symbol) [0x7ff77a6217c1]
	chromedriver!(No symbol) [0x7ff77a61f490]
	chromedriver!(No symbol) [0x7ff77a61fc02]
	chromedriver!(No symbol) [0x7ff77a638d67]
	chromedriver!(No symbol) [0x7ff77a6e0994]
	chromedriver!(No symbol) [0x7ff77a6b93ea]
	chromedriver!(No symbol) [0x7ff77a6dfbea]
	chromedriver!(No symbol) [0x7ff77a681d02]
	chromedriver!(No symbol) [0x7ff77a682b13]
	chromedriver!GetHandleVerifier [0x7ff77ad11411+4

There was an error managing chromedriver (error sending request for url (https://googlechromelabs.github.io/chrome-for-testing/known-good-versions-with-downloads.json)); using driver found in the cache


[nha_rieng] Mo trang 38: https://batdongsan.com.vn/ban-nha-rieng-tp-hcm/p38
[nha_rieng] Loi tai trang: Message: unknown error: net::ERR_INTERNET_DISCONNECTED
  (Session info: chrome=153.0.8010.52)
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff77a8f1035+5a95]
	chromedriver!(No symbol) [0x7ff77a812c70]
	chromedriver!(No symbol) [0x7ff77a634a7d]
	chromedriver!(No symbol) [0x7ff77a63186b]
	chromedriver!(No symbol) [0x7ff77a621da1]
	chromedriver!(No symbol) [0x7ff77a623be4]
	chromedriver!(No symbol) [0x7ff77a622334]
	chromedriver!(No symbol) [0x7ff77a621b0b]
	chromedriver!(No symbol) [0x7ff77a6217c1]
	chromedriver!(No symbol) [0x7ff77a61f490]
	chromedriver!(No symbol) [0x7ff77a61fc02]
	chromedriver!(No symbol) [0x7ff77a638d67]
	chromedriver!(No symbol) [0x7ff77a6e0994]
	chromedriver!(No symbol) [0x7ff77a6b93ea]
	chromedriver!(No symbol) [0x7ff77a6dfbea]
	chromedriver!(No symbol) [0x7ff77a681d02]
	chromedriver!(No symbol) [0x7ff77a682b13]
	chromedriver!GetHandleVerifier [0x7ff77ad11411+4

...

[nha_rieng] Mo trang 38: https://batdongsan.com.vn/ban-nha-rieng-tp-hcm/p38
[nha_rieng] Loi tai trang: Message: unknown error: net::ERR_INTERNET_DISCONNECTED
  (Session info: chrome=153.0.8010.52)
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff77a8f1035+5a95]
	chromedriver!(No symbol) [0x7ff77a812c70]
	chromedriver!(No symbol) [0x7ff77a634a7d]
	chromedriver!(No symbol) [0x7ff77a63186b]
	chromedriver!(No symbol) [0x7ff77a621da1]
	chromedriver!(No symbol) [0x7ff77a623be4]
	chromedriver!(No symbol) [0x7ff77a622334]
	chromedriver!(No symbol) [0x7ff77a621b0b]
	chromedriver!(No symbol) [0x7ff77a6217c1]
	chromedriver!(No symbol) [0x7ff77a61f490]
	chromedriver!(No symbol) [0x7ff77a61fc02]
	chromedriver!(No symbol) [0x7ff77a638d67]
	chromedriver!(No symbol) [0x7ff77a6e0994]
	chromedriver!(No symbol) [0x7ff77a6b93ea]
	chromedriver!(No symbol) [0x7ff77a6dfbea]
	chromedriver!(No symbol) [0x7ff77a681d02]
	chromedriver!(No symbol) [0x7ff77a682b13]
	chromedriver!GetHandleVerifier [0x7ff77ad11411+4

There was an error managing chromedriver (error sending request for url (https://googlechromelabs.github.io/chrome-for-testing/known-good-versions-with-downloads.json)); using driver found in the cache


[nha_rieng] Mo trang 38: https://batdongsan.com.vn/ban-nha-rieng-tp-hcm/p38
[nha_rieng] Loi tai trang: Message: unknown error: net::ERR_INTERNET_DISCONNECTED
  (Session info: chrome=153.0.8010.52)
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff77a8f1035+5a95]
	chromedriver!(No symbol) [0x7ff77a812c70]
	chromedriver!(No symbol) [0x7ff77a634a7d]
	chromedriver!(No symbol) [0x7ff77a63186b]
	chromedriver!(No symbol) [0x7ff77a621da1]
	chromedriver!(No symbol) [0x7ff77a623be4]
	chromedriver!(No symbol) [0x7ff77a622334]
	chromedriver!(No symbol) [0x7ff77a621b0b]
	chromedriver!(No symbol) [0x7ff77a6217c1]
	chromedriver!(No symbol) [0x7ff77a61f490]
	chromedriver!(No symbol) [0x7ff77a61fc02]
	chromedriver!(No symbol) [0x7ff77a638d67]
	chromedriver!(No symbol) [0x7ff77a6e0994]
	chromedriver!(No symbol) [0x7ff77a6b93ea]
	chromedriver!(No symbol) [0x7ff77a6dfbea]
	chromedriver!(No symbol) [0x7ff77a681d02]
	chromedriver!(No symbol) [0x7ff77a682b13]
	chromedriver!GetHandleVerifier [0x7ff77ad11411+4

There was an error managing chromedriver (error sending request for url (https://googlechromelabs.github.io/chrome-for-testing/known-good-versions-with-downloads.json)); using driver found in the cache


[nha_rieng] Mo trang 38: https://batdongsan.com.vn/ban-nha-rieng-tp-hcm/p38
[nha_rieng] Loi tai trang: Message: unknown error: net::ERR_INTERNET_DISCONNECTED
  (Session info: chrome=153.0.8010.52)
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff77a8f1035+5a95]
	chromedriver!(No symbol) [0x7ff77a812c70]
	chromedriver!(No symbol) [0x7ff77a634a7d]
	chromedriver!(No symbol) [0x7ff77a63186b]
	chromedriver!(No symbol) [0x7ff77a621da1]
	chromedriver!(No symbol) [0x7ff77a623be4]
	chromedriver!(No symbol) [0x7ff77a622334]
	chromedriver!(No symbol) [0x7ff77a621b0b]
	chromedriver!(No symbol) [0x7ff77a6217c1]
	chromedriver!(No symbol) [0x7ff77a61f490]
	chromedriver!(No symbol) [0x7ff77a61fc02]
	chromedriver!(No symbol) [0x7ff77a638d67]
	chromedriver!(No symbol) [0x7ff77a6e0994]
	chromedriver!(No symbol) [0x7ff77a6b93ea]
	chromedriver!(No symbol) [0x7ff77a6dfbea]
	chromedriver!(No symbol) [0x7ff77a681d02]
	chromedriver!(No symbol) [0x7ff77a682b13]
	chromedriver!GetHandleVerifier [0x7ff77ad11411+4

#### Đảm bảo đúng thứ tự và số lượng trường

In [6]:
columns = [
    "ma_tin",
    "tieu_de",
    "loai_hinh",
    "gia",
    "dien_tich",
    "so_phong_ngu",
    "so_tang",
    "huong_nha",
    "tinh_thanh",
    "quan_huyen",
    "phuong_xa",
    "tinh_trang_phap_ly",
    "ngay_dang",
    "ten_moi_gioi_san",
    "nguon",
    "khu_vuc",
    "dia_diem",
    "mo_ta",
    "nguoi_dang",
    "thoi_gian_dang",
    "so_phong",
    "url"
]


def order_record(record):
    return {
        column: record.get(column)
        for column in columns
    }


df_result = [
    order_record(record)
    for record in df_result
]

print("So cot:", len(columns))
print(columns)

So cot: 22
['ma_tin', 'tieu_de', 'loai_hinh', 'gia', 'dien_tich', 'so_phong_ngu', 'so_tang', 'huong_nha', 'tinh_thanh', 'quan_huyen', 'phuong_xa', 'tinh_trang_phap_ly', 'ngay_dang', 'ten_moi_gioi_san', 'nguon', 'khu_vuc', 'dia_diem', 'mo_ta', 'nguoi_dang', 'thoi_gian_dang', 'so_phong', 'url']


### Bước 7: Kiểm tra dữ liệu

In [7]:
print("===== KICH THUOC DATASET =====")
print((len(df_result), len(columns)))

print("\n===== CAC COT =====")
print(columns)

print("\n===== SO LUONG DU LIEU THIEU =====")
missing_values = {
    column: sum(
        record.get(column) in (None, "")
        for record in df_result
    )
    for column in columns
}
print(missing_values)

print("\n===== TRUNG LAP MA TIN =====")
ma_tin_values = [record.get("ma_tin") for record in df_result]
print(len(ma_tin_values) - len(set(ma_tin_values)))

print("\n===== TRUNG LAP URL =====")
url_values = [record.get("url") for record in df_result]
print(len(url_values) - len(set(url_values)))

===== KICH THUOC DATASET =====
(5000, 22)

===== CAC COT =====
['ma_tin', 'tieu_de', 'loai_hinh', 'gia', 'dien_tich', 'so_phong_ngu', 'so_tang', 'huong_nha', 'tinh_thanh', 'quan_huyen', 'phuong_xa', 'tinh_trang_phap_ly', 'ngay_dang', 'ten_moi_gioi_san', 'nguon', 'khu_vuc', 'dia_diem', 'mo_ta', 'nguoi_dang', 'thoi_gian_dang', 'so_phong', 'url']

===== SO LUONG DU LIEU THIEU =====
{'ma_tin': 0, 'tieu_de': 0, 'loai_hinh': 0, 'gia': 95, 'dien_tich': 0, 'so_phong_ngu': 3315, 'so_tang': 4277, 'huong_nha': 3994, 'tinh_thanh': 4612, 'quan_huyen': 1878, 'phuong_xa': 633, 'tinh_trang_phap_ly': 4212, 'ngay_dang': 5, 'ten_moi_gioi_san': 1906, 'nguon': 0, 'khu_vuc': 1878, 'dia_diem': 5000, 'mo_ta': 0, 'nguoi_dang': 5000, 'thoi_gian_dang': 5, 'so_phong': 5000, 'url': 0}

===== TRUNG LAP MA TIN =====
12

===== TRUNG LAP URL =====
10


### Bước 8: Xem toàn bộ dataset

In [8]:
for record in df_result:
    print(record)

{'ma_tin': '46241488', 'tieu_de': 'Cập nhật giỏ hàng giá tốt (1PN -6tỷ5) (2PN-8tỷ8) (3PN -13tỷ5) (4PN -19tỷ) Vinhomes Central Park', 'loai_hinh': 'can_ho_chung_cu', 'gia': '19tỷ', 'dien_tich': '120 m²', 'so_phong_ngu': '1', 'so_tang': None, 'huong_nha': 'Tây', 'tinh_thanh': None, 'quan_huyen': None, 'phuong_xa': 'P. Thạnh Mỹ Tây mới', 'tinh_trang_phap_ly': None, 'ngay_dang': '16/09/2026', 'ten_moi_gioi_san': None, 'nguon': 'batdongsan.com.vn', 'khu_vuc': None, 'dia_diem': None, 'mo_ta': 'Cập nhật giỏ hàng giá tốt (1PN -6tỷ5) (2PN-8tỷ8) (3PN -13tỷ5) (4PN -19tỷ) Vinhomes Central Park 13,5 tỷ · 120 m² · 112,5 tr/m² · 3 · 2 · Q. Bình Thạnh (P. Thạnh Mỹ Tây mới) Giỏ hàng căn hộ Vinhomes Central Park giá tham khảo tốt trong khu vực 1PN - 2PN - 3PN - 4PN duplex penthouse. Giá bán & diện tích căn hộ: - 1 phòng ngủ: - diện tích (36m² - 55m²). - Giá bán chỉ: 6tỷ đến 7tỷ5 full nội thất decor. Tư vấn miễn phí gọi ngay: 0909 638 *** .* Căn 2 phòng ngủ. (65m² - 89m²). - Giá bán chỉ: 8tỷ8 đến 11tỷ5 (

### Bước 9: Lưu dataset

In [9]:
with open(
    OUTPUT_FILE,
    "w",
    newline="",
    encoding="utf-8-sig"
) as csv_file:
    writer = csv.DictWriter(
        csv_file,
        fieldnames=columns
    )
    writer.writeheader()
    writer.writerows(df_result)

print("DA LUU:", OUTPUT_FILE)


DA LUU: batdongsan_hcm.csv
